# 04 - Avaliação e teste entre bases

Métricas por classe e por tamanho de E1 e E2 (`configs/avaliacao.yaml`,
[detalhamento](../docs/fases/fase-4.md)). **Cada teste roda uma única vez.**

**No Colab, com GPU T4**, em duas passagens:

1. Execute tudo. O desenvolvimento é reproduzido, os limiares são escolhidos e a execução
   **para** antes do teste.
2. Grave os dois limiares em `configs/avaliacao.yaml`, publique o commit, reinicie o
   ambiente e execute tudo de novo. O teste roda; o que já terminou é pulado.
3. Baixe `epi-sistema/f4/entrega.zip` do Drive.

Fora do Colab o modo é **curto**: só desenvolvimento, com as listas reduzidas do notebook 01,
para conferir o código. Os números não valem.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/Guilh3rme/epi-sistema.git"

if "COLAB_RELEASE_TAG" in os.environ:
    raiz = Path("/content/epi-sistema")
    if raiz.exists():  # traz os limiares publicados depois do clone
        subprocess.run(["git", "-C", str(raiz), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(raiz)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))  # dispensa reiniciar o ambiente
else:
    raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

(raiz / "runs").mkdir(exist_ok=True)
os.chdir(raiz / "runs")  # pesos e resultados ficam em runs/, fora do git
print("raiz:", raiz.name)

In [ ]:
from epi.ambiente import configurar_ultralytics, em_colab

configurar_ultralytics(raiz)  # antes de qualquer importação do ultralytics
import json
import shutil

import torch

from epi import avaliacao, dados, treino

MODO = os.environ.get("EPI_MODO") or ("completo" if em_colab() else "curto")
assert MODO in ("curto", "completo"), MODO
COMPLETO = MODO == "completo"
SPLIT = "test" if COMPLETO else "val"  # o modo curto nunca abre o teste
cfg = dados.carregar_config(raiz / "configs" / "avaliacao.yaml")
cfg_treino = dados.carregar_config(raiz / "configs" / "treino.yaml")
nomes = {e: treino.nome_execucao(e, cfg_treino) for e in ("e1", "e2")}
if COMPLETO:
    assert em_colab(), "a avaliação completa roda no Colab"
    assert torch.cuda.is_available(), "selecione um ambiente com GPU T4"
    from google.colab import drive

    drive.mount("/content/drive")
    meu_drive = Path("/content/drive/MyDrive/epi-sistema")
    saida, copia = raiz / "runs" / "f4", meu_drive / "f4"
    preparado, divisoes = raiz / "data" / "preparado", raiz / "configs" / "divisoes"
    pesos = {e: meu_drive / "f3" / n / "weights" / "best.pt" for e, n in nomes.items()}
    referencia = {e: json.loads((raiz / "experimentos" / "f3" / n / "resumo.json").read_text(
        encoding="utf-8")) for e, n in nomes.items()}
    gpu = torch.cuda.get_device_name(0)
    gpus_f3 = {r["sessoes"][-1]["gpu"] for r in referencia.values()}
    assert gpus_f3 == {gpu}, f"use a mesma GPU da F3 ({gpus_f3}), não {gpu}"
else:
    saida = raiz / "runs" / "f4_curto"
    shutil.rmtree(saida, ignore_errors=True)  # nada aqui é resultado
    copia = saida / "copia"
    preparado, divisoes = raiz / "data" / "preparado_curto", raiz / "runs" / "f2_curto" / "divisoes"
    pesos = {e: raiz / "runs" / "pesos" / f"{n}.pt" for e, n in nomes.items()}
    referencia = dict.fromkeys(nomes)
    cfg["principais"] = ["pessoa", "luvas"]  # a amostra local não tem caixas das quatro
copia.mkdir(parents=True, exist_ok=True)
device = 0 if torch.cuda.is_available() else "cpu"
commit = subprocess.run(["git", "-C", str(raiz), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"MODO: {MODO} | divisão: {SPLIT} | commit: {commit} | "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")

## Bases preparadas e divisões congeladas

In [ ]:
if COMPLETO and not all((preparado / f"{b}.json").exists() for b in ("sh17", "cppe")):
    dados.restaurar(meu_drive / "f2" / "bases_preparadas.tar", preparado.parent)
dados.verificar_divisoes(divisoes)
print("divisões conferidas")

## Desenvolvimento: reprodução da F3 e limiares

Nenhum conjunto de teste é lido aqui.

In [ ]:
dev = {e: avaliacao.desenvolvimento(e, cfg, cfg_treino, preparado, divisoes, saida, copia,
                                    pesos[e], referencia[e], device=device) for e in nomes}
for e, r in dev.items():
    print(f"LIMIAR {e}: {r['limiar']:.6f} (F1 médio {r['f1_medio']:.3f}, "
          f"diferença para a F3: AP {r['maior_diferenca_ap']}, P e R {r['maior_diferenca_pr']})")

## PARADA antes do teste

Grave os valores `LIMIAR` acima em `configs/avaliacao.yaml`, publique e execute tudo de novo.

In [ ]:
if COMPLETO:
    faltam = [e for e in nomes if cfg["limiares"][e] is None]
    if faltam:
        drive.flush_and_unmount()  # sem isto o desenvolvimento pode não chegar ao Drive
    assert not faltam, f"PARADA: grave e publique os limiares de {faltam}"
print("pode avaliar:", SPLIT)

## Avaliação

In [ ]:
for execucao, (e, *_) in avaliacao.EXECUCOES.items():
    r = avaliacao.avaliar(execucao, SPLIT, cfg, cfg_treino, preparado, divisoes, saida, copia,
                          pesos[e], device=device)
    print(f"CONCLUÍDA {execucao}: {r['imagens']} imagens ({r['split']})")

## Tabelas e entrega

In [ ]:
from IPython.display import Markdown, display

avaliacao.tabelas(copia, copia / "tabelas", cfg)
for nome in ("por_classe", "por_tamanho"):
    display(Markdown((copia / "tabelas" / f"{nome}.md").read_text(encoding="utf-8")))
arquivo = avaliacao.entregar(copia, saida / "entrega.zip")
shutil.copyfile(arquivo, copia / "entrega.zip")
print("entrega:", copia / "entrega.zip")

In [ ]:
if COMPLETO:
    drive.flush_and_unmount()  # garante o envio ao Drive antes de desconectar
    print("Drive sincronizado: já pode desconectar")
print("avaliação OK" if COMPLETO else "teste curto OK (os números não valem)")